In [ ]:
# coding: utf-8
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import copy, csv, json, random, time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import swin_b

DEVICE     = "cuda"
SEED       = 42
UEC_ROOT   = Path("/root/data/UECFood256/UECFOOD256")
ALIGN_CKPT = "/workspace/align/keep/align_train_20k_seed42.pth"
ARMS       = Path("/workspace/uec256_lt_arms"); ARMS.mkdir(exist_ok=True)
POOL       = Path("/workspace/uec256_synth");   POOL.mkdir(exist_ok=True)
META       = Path("/workspace/uec256_meta");    META.mkdir(exist_ok=True)

EPOCHS, BATCH, LR = 30, 32, 1e-4
WARMUP, WD, MIXUP, SMOOTH, EMA_DECAY, TTA_VIEWS, WORKERS = 3, 1e-4, 0.2, 0.1, 0.999, 8, 12

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

free, total = torch.cuda.mem_get_info()
print(f"torch {torch.__version__}   {free/1e9:.1f} GB free of {total/1e9:.1f}")

In [ ]:
def build_instances(root):
    root = Path(root)
    cats = sorted([d for d in root.iterdir() if d.is_dir() and d.name.isdigit()],
                  key=lambda d: int(d.name))
    c2i = {d.name: i for i, d in enumerate(cats)}
    out = []
    for d in cats:
        bb = d/"bb_info.txt"
        if not bb.exists(): continue
        for line in bb.read_text().splitlines()[1:]:
            p = line.split()
            if len(p) != 5: continue
            img_id, x1, y1, x2, y2 = p
            f = d/f"{img_id}.jpg"
            box = (int(x1), int(y1), int(x2), int(y2))
            if f.exists() and box[2] > box[0] and box[3] > box[1]:
                out.append((f, box, c2i[d.name], img_id))
    return out, len(cats), cats

def group_split(inst, seed=SEED):
    rng = random.Random(seed)
    by_stem = defaultdict(list)
    for x in inst: by_stem[x[3]].append(x)
    stems_by_cat = defaultdict(list)
    for stem, items in by_stem.items():
        stems_by_cat[min(i[2] for i in items)].append(stem)
    tr, va, te = [], [], []
    for cat in sorted(stems_by_cat):
        s = stems_by_cat[cat][:]; rng.shuffle(s)
        a, b = int(.70*len(s)), int(.85*len(s))
        for x in s[:a]: tr += by_stem[x]
        for x in s[a:b]: va += by_stem[x]
        for x in s[b:]: te += by_stem[x]
    assert not ({i[3] for i in tr} & {i[3] for i in te}), "leakage across split"
    return tr, va, te

MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
    transforms.ColorJitter(0.3,0.3,0.3,0.05),
    transforms.ToTensor(), transforms.Normalize(MEAN,STD),
    transforms.RandomErasing(p=0.25)])
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                              transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
tta_tf  = transforms.Compose([transforms.Resize(256), transforms.RandomCrop(224),
                              transforms.RandomHorizontalFlip(),
                              transforms.ToTensor(), transforms.Normalize(MEAN,STD)])

class UEC(Dataset):
    """Real instances are (path, box, label, stem). Synthetic are (path, None, label, None)."""
    def __init__(self, inst, tf, synth=None):
        self.inst = list(inst)
        self.n_real = len(self.inst)
        self.n_synth = 0
        if synth:
            self.inst += synth; self.n_synth = len(synth)
        self.tf = tf
    def __len__(self): return len(self.inst)
    def __getitem__(self, i):
        f, box, y, _ = self.inst[i]
        im = Image.open(f).convert("RGB")
        if box is not None: im = im.crop(box)
        return self.tf(im), y

def load_synth_list(path):
    if not path: return None
    out = []
    for ln in Path(path).read_text().splitlines():
        if not ln.strip(): continue
        img, cid = ln.split("\t")
        out.append((Path(img), None, int(cid), None))
    return out

seed_all()
inst, N_CLASSES, CAT_DIRS = build_instances(UEC_ROOT)
TR_INST, VA_INST, TE_INST = group_split(inst)

# class names, indexed to match c2i (0-based, ordered by numeric dir name)
_names = {}
for ln in (UEC_ROOT/"category.txt").read_text().splitlines()[1:]:
    p = ln.split("\t")
    if len(p) >= 2: _names[int(p[0])] = p[1].strip()
CLASSES = [_names.get(int(d.name), f"class_{d.name}") for d in CAT_DIRS]

train_counts = np.bincount([x[2] for x in TR_INST], minlength=N_CLASSES)

mk = lambda d, s: DataLoader(d, batch_size=BATCH, shuffle=s,
                             num_workers=WORKERS, pin_memory=True)
va_dl  = mk(UEC(VA_INST, eval_tf), False)
te_dl  = mk(UEC(TE_INST, eval_tf), False)
tta_dl = mk(UEC(TE_INST, tta_tf), False)

print(f"{len(inst)} instances -> {len(TR_INST)} train / {len(VA_INST)} val / "
      f"{len(TE_INST)} test, {N_CLASSES} classes")
print(f"train per class: min {train_counts.min()}  median {int(np.median(train_counts))}  "
      f"max {train_counts.max()}")
print(f"example classes: {CLASSES[:3]} ... {CLASSES[-2:]}")

In [ ]:
def build_swin(n_classes, init):
    m = swin_b(weights="IMAGENET1K_V1" if init == "imagenet" else None)
    dim = m.head.in_features
    m.head = nn.Identity()
    if init == "align":
        sd = torch.load(ALIGN_CKPT, map_location="cpu")
        sd = sd.get("state_dict", sd.get("model", sd))
        ext = {k[len("visual."):]: v for k, v in sd.items() if k.startswith("visual.")}
        missing, _ = m.load_state_dict(ext, strict=False)
        loaded = len(m.state_dict()) - len(missing)
        print(f"alignment init: {loaded}/{len(m.state_dict())} tensors")
        assert loaded > 0.8*len(m.state_dict()), "prefix mismatch"
    m.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(dim, n_classes))
    return m

class EMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()):
            s.mul_(self.decay).add_(m.detach(), alpha=1-self.decay)
        for s, m in zip(self.shadow.buffers(), model.buffers()):
            s.copy_(m)

def mixup(x, y, alpha=MIXUP):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

In [ ]:
@torch.no_grad()
def evaluate(model, loader, want_cm=False):
    model.eval(); c1, c5, ls, pr = [], [], [], []
    for x, y in loader:
        top5 = model(x.to(DEVICE, non_blocking=True)).topk(5, 1, True, True)[1].cpu()
        cc = top5.eq(y.view(-1,1))
        c1.append(cc[:,0]); c5.append(cc.any(1)); ls.append(y); pr.append(top5[:,0])
    c1, c5, labels = torch.cat(c1), torch.cat(c5), torch.cat(ls)
    out = {"top1": round(100*c1.float().mean().item(), 2),
           "top5": round(100*c5.float().mean().item(), 2)}
    if not want_cm: return out, None
    cm = np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64)
    np.add.at(cm, (labels.numpy(), torch.cat(pr).numpy()), 1)
    return out, cm

@torch.no_grad()
def evaluate_tta(model, base, tta, n_views=TTA_VIEWS, want_cm=False):
    model.eval(); probs = labels = None
    for v in range(n_views):
        loader = base if v == 0 else tta
        ps, ls = [], []
        for x, y in loader:
            ps.append(F.softmax(model(x.to(DEVICE, non_blocking=True)), 1).cpu()); ls.append(y)
        vp = torch.cat(ps)
        probs = vp if probs is None else probs + vp
        if labels is None: labels = torch.cat(ls)
    top5 = probs.topk(5, 1, True, True)[1]
    cc = top5.eq(labels.view(-1,1))
    out = {"top1": round(100*cc[:,0].float().mean().item(), 2),
           "top5": round(100*cc.any(1).float().mean().item(), 2)}
    if not want_cm: return out, None
    cm = np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64)
    np.add.at(cm, (labels.numpy(), top5[:,0].numpy()), 1)
    return out, cm

In [ ]:
def run_arm(tag, init="align", synth=None, alloc="none",
            expect_synth=None, seed=SEED):
    seed_all(seed)
    tr = UEC(TR_INST, train_tf, load_synth_list(synth))
    if expect_synth is not None and tr.n_synth != expect_synth:
        raise SystemExit(f"budget mismatch: {tr.n_synth} synthetic, expected {expect_synth}")
    tr_dl = mk(tr, True)
    print(f"{tag}:  real {tr.n_real}   synth {tr.n_synth}   total {len(tr)}   seed {seed}")

    model = build_swin(N_CLASSES, init).to(DEVICE)
    ema   = EMA(model)
    crit  = nn.CrossEntropyLoss(label_smoothing=SMOOTH)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.SequentialLR(opt,
        [torch.optim.lr_scheduler.LinearLR(opt, 0.1, 1.0, WARMUP),
         torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS-WARMUP)],
        milestones=[WARMUP])
    scaler = torch.amp.GradScaler("cuda")

    best, hist, t0 = -1.0, [], time.time()
    ck = ARMS/f"{tag}.pth"
    for ep in range(EPOCHS):
        model.train()
        for x, y in tr_dl:
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            xm, ya, yb, lam = mixup(x, y)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                o = model(xm)
                loss = lam*crit(o, ya) + (1-lam)*crit(o, yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            ema.update(model)
        sched.step()
        r, _ = evaluate(model, va_dl)
        e, _ = evaluate(ema.shadow, va_dl)
        hist.append({"epoch": ep+1, "raw_top1": r["top1"], "ema_top1": e["top1"]})
        if max(r["top1"], e["top1"]) > best:
            best = max(r["top1"], e["top1"])
            torch.save(ema.shadow.state_dict() if e["top1"] >= r["top1"]
                       else model.state_dict(), ck)
        print(f"  ep {ep+1:>2}/{EPOCHS}  raw {r['top1']:.2f}  ema {e['top1']:.2f}  "
              f"best {best:.2f}  ({time.time()-t0:.0f}s)", flush=True)

    model.load_state_dict(torch.load(ck, map_location=DEVICE)); model.to(DEVICE)
    no_tta,  cm  = evaluate(model, te_dl, want_cm=True)
    with_tta, cmt = evaluate_tta(model, te_dl, tta_dl, want_cm=True)
    np.save(ARMS/f"{tag}_confusion.npy", cm)
    np.save(ARMS/f"{tag}_confusion_tta.npy", cmt)

    res = {"tag": tag, "init": init, "alloc": alloc, "synth_list": str(synth),
           "n_real": tr.n_real, "n_synth": tr.n_synth, "n_classes": N_CLASSES,
           "config": {"epochs": EPOCHS, "batch": BATCH, "lr": LR, "warmup": WARMUP,
                      "wd": WD, "mixup": MIXUP, "smooth": SMOOTH, "ema": EMA_DECAY,
                      "tta_views": TTA_VIEWS, "seed": seed},
           "no_tta": no_tta, "with_tta": with_tta, "best_val": best,
           "minutes": round((time.time()-t0)/60, 1), "history": hist}
    (ARMS/f"{tag}_results.json").write_text(json.dumps(res, indent=2))

    print(f"\n{tag}")
    print(f"  no TTA       top1 {no_tta['top1']:6.2f}   top5 {no_tta['top5']:6.2f}")
    print(f"  8-view TTA   top1 {with_tta['top1']:6.2f}   top5 {with_tta['top5']:6.2f}")
    print(f"  {res['minutes']} min\n")
    return res

In [ ]:
for s in (42, 1, 2):
    run_arm(f"A_align_s{s}", init="align", alloc="none", seed=s)

In [ ]:
cm = np.load(ARMS/"A_align_s42_confusion.npy")
row_err = cm.sum(axis=1) - np.diag(cm)
total_err = int(cm.sum() - np.trace(cm))
print(f"test instances {cm.sum()}   errors {total_err}  ({100*total_err/cm.sum():.1f}%)\n")

sym = cm + cm.T
np.fill_diagonal(sym, 0)
iu = np.triu_indices(N_CLASSES, k=1)
pair = sym[iu]
order = np.argsort(-pair)

print(f"{'#':>3}  {'class A':<28} {'class B':<28} {'err':>4}  A>B/B>A")
for n, k in enumerate(order[:25], 1):
    i, j = int(iu[0][k]), int(iu[1][k])
    print(f"{n:>3}  {CLASSES[i][:28]:<28} {CLASSES[j][:28]:<28} {pair[k]:>4}  "
          f"{cm[i,j]:>3}/{cm[j,i]:<3}")

for kk in (10, 25, 50, 100, 200):
    print(f"\ntop-{kk:>3} pairs cover {100*pair[order[:kk]].sum()/total_err:5.1f}% of errors")

print(f"\nclasses with zero errors: {(row_err==0).sum()}")
print(f"error concentration — top 25 classes hold "
      f"{100*np.sort(row_err)[-25:].sum()/total_err:.1f}% of errors")

In [ ]:
val_dls = {}
cms = []
for s in (42, 1, 2):
    m = build_swin(N_CLASSES, "align").to(DEVICE)
    m.load_state_dict(torch.load(ARMS/f"A_align_s{s}.pth", map_location=DEVICE))
    _, cm_v = evaluate(m, va_dl, want_cm=True)
    cms.append(cm_v)
    print(f"seed {s}: {int(cm_v.sum()-np.trace(cm_v))} val errors")
    del m; torch.cuda.empty_cache()

cm_val = sum(cms)
np.save(META/"A_align_val_confusion_pooled.npy", cm_val)
row_err_val = cm_val.sum(axis=1) - np.diag(cm_val)
tot = int(cm_val.sum() - np.trace(cm_val))
print(f"\npooled over 3 seeds: {tot} errors, {tot/N_CLASSES:.1f} per class")
print(f"classes with 0 errors: {(row_err_val==0).sum()}")
print(f"classes with <3 errors: {(row_err_val<3).sum()}")

sym = cm_val + cm_val.T; np.fill_diagonal(sym, 0)
iu = np.triu_indices(N_CLASSES, k=1); pair = sym[iu]
order = np.argsort(-pair)
print(f"\ntop-100 pairs cover {100*pair[order[:100]].sum()/tot:.1f}% of errors")
print(f"\n{'#':>3}  {'class A':<28} {'class B':<28} {'err':>4}")
for n, k in enumerate(order[:15], 1):
    i, j = int(iu[0][k]), int(iu[1][k])
    print(f"{n:>3}  {CLASSES[i][:28]:<28} {CLASSES[j][:28]:<28} {pair[k]:>4}")

In [ ]:
prompts = {}
for i, c in enumerate(CLASSES):
    prompts[c] = c          # placeholder: bare class name
Path(META/"prompts_draft.json").write_text(json.dumps(prompts, indent=2, ensure_ascii=False))
print(f"wrote {META/'prompts_draft.json'} with {len(prompts)} entries")
for c in CLASSES[:20]: print(f"  {c}")

In [ ]:
from collections import Counter
dupes = [c for c, n in Counter(CLASSES).items() if n > 1]
print("duplicate names:", dupes)
for d in dupes:
    print([i for i, c in enumerate(CLASSES) if c == d])

In [ ]:
prompts = {str(i): {"name": c, "prompt": c} for i, c in enumerate(CLASSES)}
Path(META/"prompts_draft.json").write_text(
    json.dumps(prompts, indent=2, ensure_ascii=False))
print(f"wrote {len(prompts)} entries")

In [ ]:
{
  "0":  "a bowl of plain steamed white Japanese rice",
  "1":  "unadon, grilled eel glazed with sweet soy sauce served over rice in a lacquered box",
  "2":  "pilaf, seasoned rice pilaf cooked with vegetables and stock",
  "3":  "oyakodon, simmered chicken and soft-set egg over rice in a donburi bowl",
  "4":  "katsudon, breaded deep-fried pork cutlet with egg and onion over rice in a bowl",
  "5":  "Japanese beef curry, thick brown curry sauce with beef served beside white rice on a plate",
  "6":  "nigiri sushi, slices of raw fish on small oblong beds of vinegared rice",
  "7":  "Hainanese chicken rice, poached sliced chicken with seasoned rice, cucumber and chilli sauce",
  "8":  "chahan, Chinese-style fried rice stir-fried with egg, spring onion and small pieces of pork",
  "9":  "tendon, assorted tempura pieces glazed with sweet sauce served over rice in a bowl",
  "10": "bibimbap, Korean rice bowl topped with arranged namul vegetables, beef, fried egg and gochujang",
  "11": "thick slice of Japanese shokupan toast, lightly browned, on a plate",
  "12": "a plain butter croissant on a plate",
  "13": "Japanese roll bread, small soft white dinner rolls",
  "14": "raisin bread, sliced soft white loaf studded with raisins",
  "15": "chip butty, thick-cut chips in a buttered white bread sandwich",
  "16": "a hamburger with beef patty, lettuce, tomato and cheese in a sesame seed bun",
  "17": "a whole round pizza with melted mozzarella and tomato sauce",
  "18": "sandwiches, triangular crustless white bread sandwiches on a plate",
  "19": "udon, thick white wheat noodles in clear dashi broth in a bowl"
}

In [ ]:
for i in range(20, 60):
    print(f"{i}: {CLASSES[i]}")

In [ ]:
from collections import Counter
dupes = [c for c, n in Counter(CLASSES).items() if n > 1]
print("duplicates:", dupes)
for d in dupes:
    print(d, "->", [i for i, c in enumerate(CLASSES) if c == d])

In [ ]:
for i in (39, 66):
    print(i, CLASSES[i], "| train instances:", train_counts[i])
print("confused with each other:", cm_val[39,66] + cm_val[66,39], "val errors")

In [ ]:
{
  "20": "tempura udon, thick white udon noodles in dashi broth topped with a large battered prawn tempura",
  "21": "soba, thin greyish-brown buckwheat noodles in dark dashi broth in a bowl",
  "22": "tonkotsu ramen, thin wheat ramen noodles in pale pork bone broth with chashu pork, egg and nori",
  "23": "Taiwanese beef noodle soup, wide noodles in dark red braised beef broth with chunks of stewed beef and greens",
  "24": "tenshin-men, ramen noodles in clear broth topped with a large crab and egg omelette",
  "25": "yakisoba, pan-fried wheat noodles stir-fried with cabbage, pork and brown sauce, no soup",
  "26": "spaghetti with tomato sauce on a plate, twirled long pasta",
  "27": "okonomiyaki, thick savoury Japanese pancake with cabbage, brown sauce, mayonnaise stripes and bonito flakes",
  "28": "takoyaki, round ball-shaped octopus dumplings in a tray with sauce, mayonnaise and dancing bonito flakes",
  "29": "gratin, creamy white sauce baked with browned cheese crust in an oval ceramic dish",
  "30": "yasai itame, stir-fried mixed vegetables with cabbage, carrot and bean sprouts on a plate",
  "31": "korokke, oval deep-fried breadcrumbed potato croquettes with golden crust on a plate",
  "32": "yakinasu, grilled Japanese eggplant, charred and peeled, topped with grated ginger and soy",
  "33": "sauteed spinach, wilted dark green spinach leaves lightly fried, served in a small dish",
  "34": "vegetable-only tempura, battered deep-fried sweet potato, pumpkin, aubergine and shiso leaf, no seafood",
  "35": "miso soup, plain light brown miso broth with small cubes of tofu and wakame seaweed in a lacquered bowl",
  "36": "potage, smooth thick creamy pale soup in a wide white bowl",
  "37": "sausage, grilled pork sausages on a plate",
  "38": "oden, assorted simmered items in clear dashi broth — daikon, boiled egg, fish cake, konnyaku triangle",
  "39": "tamagoyaki-style plain omelette, folded yellow egg omelette on a plate with no filling visible",
  "40": "ganmodoki, round deep-fried tofu fritters with carrot and burdock specks, simmered in broth",
  "41": "gyoza, pan-fried crescent dumplings with browned crisp bottoms arranged on a plate",
  "42": "beef stew, dark brown stew with chunks of beef, carrot and potato in a bowl",
  "43": "teriyaki grilled fish, fish fillet glazed with glossy dark teriyaki sauce",
  "44": "deep-fried white fish fillet with golden breadcrumb coating on a plate",
  "45": "grilled salmon fillet, orange-pink flesh with crisp skin and grill marks",
  "46": "salmon meuniere, salmon fillet pan-fried in butter and flour with lemon and parsley butter sauce",
  "47": "sashimi, thick slices of raw tuna and white fish on a plate with shiso leaf and grated daikon, no rice",
  "48": "sanma shioyaki, whole grilled pacific saury, a long silver fish with charred skin, served with grated daikon",
  "49": "sukiyaki, thinly sliced beef, tofu, shiitake and shirataki simmered in a shallow iron pot in sweet soy broth",
  "50": "subuta, sweet and sour pork in glossy orange-red sauce with green pepper, onion and pineapple",
  "51": "tataki, lightly seared fish with a raw pink centre, sliced and arranged on a plate",
  "52": "chawanmushi, savoury steamed egg custard, smooth pale yellow surface in a small lidded cup",
  "53": "assorted tempura including large battered prawns and vegetables, arranged on paper on a plate",
  "54": "karaage, Japanese bite-sized deep-fried chicken pieces, craggy golden coating, with lemon wedge",
  "55": "sirloin cutlet, breaded deep-fried beef sirloin sliced into strips",
  "56": "nanbanzuke, deep-fried small fish marinated in sweet vinegar with sliced onion, carrot and red chilli",
  "57": "nizakana, fish simmered whole in sweet soy broth, dark glossy, served in a shallow bowl",
  "58": "nikujaga, simmered beef and potato with onion and carrot in sweet soy broth",
  "59": "hambagu, Japanese hamburger steak patty with brown demi-glace sauce on a plate, no bun"
}

In [ ]:
for i in range(60, 100):
    print(f"{i}: {CLASSES[i]}")

In [ ]:
for i in (39, 66):
    print(i, repr(CLASSES[i]), "train:", train_counts[i])
print("39<->66 val errors:", cm_val[39,66] + cm_val[66,39])

In [ ]:
import matplotlib.pyplot as plt
for ci in (39, 66):
    ex = [x for x in TR_INST if x[2] == ci][:4]
    fig, ax = plt.subplots(1, 4, figsize=(12,3))
    for a, (f, box, _, _) in zip(ax, ex):
        a.imshow(Image.open(f).convert("RGB").crop(box)); a.axis("off")
    fig.suptitle(f"class {ci}: {CLASSES[ci]}  (dir {CAT_DIRS[ci].name})")
    plt.show()

In [ ]:
{
  "39": "a Western-style omelette, a large folded oval of soft pale yellow egg on a plate with a spoonful of ketchup or tomato sauce",
  "66": "tamagoyaki, Japanese rolled omelette cut into thick rectangular blocks showing spiral egg layers, arranged in a bento box"
}

In [ ]:
for i in range(100, 140):
    print(f"{i}: {CLASSES[i]}")

In [ ]:
for ci in (1, 133):
    ex = [x for x in TR_INST if x[2] == ci][:4]
    fig, ax = plt.subplots(1, 4, figsize=(12,3))
    for a, (f, box, _, _) in zip(ax, ex):
        a.imshow(Image.open(f).convert("RGB").crop(box)); a.axis("off")
    fig.suptitle(f"{ci}: {CLASSES[ci]}"); plt.show()
print("1<->133 val errors:", cm_val[1,133] + cm_val[133,1])

In [ ]:
{
  "1":   "unaju, grilled eel glazed with kabayaki sauce laid in neat parallel strips filling a rectangular black lacquered box, viewed from above",
  "133": "unadon, one or two large glazed grilled eel fillets on a mound of white rice in a round ceramic donburi bowl, white rice visible around the edges"
}

In [ ]:
SUSPECT = [(75, 76), (138, 139), (34, 53), (35, 88), (88, 89)]
for i, j in SUSPECT:
    print(f"{i} {CLASSES[i]:<32} <-> {j} {CLASSES[j]:<32} "
          f"val errors {cm_val[i,j] + cm_val[j,i]}")

In [ ]:
for pair in [(75,76), (138,139)]:
    for ci in pair:
        ex = [x for x in TR_INST if x[2] == ci][:4]
        fig, ax = plt.subplots(1, 4, figsize=(12,3))
        for a, (f, box, _, _) in zip(ax, ex):
            a.imshow(Image.open(f).convert("RGB").crop(box)); a.axis("off")
        fig.suptitle(f"{ci}: {CLASSES[ci]}"); plt.show()

In [ ]:
{
  "75":  "kaisendon, thick slices of raw tuna, salmon and white fish completely covering a bowl of rice, rice barely visible, with wasabi and shiso",
  "76":  "chirashizushi, a shallow bowl of white vinegared rice topped with shredded nori strips, thin yellow egg ribbon and small scattered toppings, mostly rice with no large fish slices",
  "138": "tonkatsu, one large flat breaded deep-fried pork loin slab sliced crosswise into strips, served on shredded cabbage",
  "139": "hirekatsu, several small round breaded deep-fried pork fillet medallions arranged together on a plate"
}

In [ ]:
for i in range(140, 180):
    print(f"{i}: {CLASSES[i]}")

In [ ]:
print("23<->172 val errors:", cm_val[23,172] + cm_val[172,23])
for ci in (23, 172):
    ex = [x for x in TR_INST if x[2] == ci][:4]
    fig, ax = plt.subplots(1, 4, figsize=(12,3))
    for a, (f, box, _, _) in zip(ax, ex):
        a.imshow(Image.open(f).convert("RGB").crop(box)); a.axis("off")
    fig.suptitle(f"{ci}: {CLASSES[ci]}"); plt.show()

In [ ]:
{
  "22": "ramen in a bowl of clear brown shoyu broth with straight thin noodles, chashu pork, menma bamboo shoots, spring onion and a sheet of nori",
  "23": "tonkotsu ramen in a bowl of thick creamy pale white pork bone broth, with chashu pork slices, a halved soft-boiled ajitama egg and spring onion"
}

In [ ]:
for i in range(180, 220):
    print(f"{i}: {CLASSES[i]}")

In [ ]:
for i in range(220, 256):
    print(f"{i}: {CLASSES[i]}")

In [ ]:
FULL = {}   # paste all seven batches in here, keyed by index string

assert len(FULL) == 256, len(FULL)
assert all(str(i) in FULL for i in range(256)), "missing indices"

out = {str(i): {"name": CLASSES[i], "prompt": FULL[str(i)]} for i in range(256)}
(META/"prompts.json").write_text(json.dumps(out, indent=2, ensure_ascii=False))
print(f"wrote {META/'prompts.json'}")

# sanity: no two prompts identical, and none is just the bare class name
from collections import Counter
dupes = [p for p, n in Counter(FULL.values()).items() if n > 1]
print("duplicate prompts:", len(dupes))
bare = [i for i in range(256) if FULL[str(i)].strip().lower() == CLASSES[i].strip().lower()]
print("bare-name prompts remaining:", bare)

In [ ]:
FULL = json.loads((META/"uec256_prompts.json").read_text())

assert len(FULL) == 256
assert all(str(i) in FULL for i in range(256))

out = {str(i): {"name": CLASSES[i], "prompt": FULL[str(i)]} for i in range(256)}
(META/"prompts.json").write_text(json.dumps(out, indent=2, ensure_ascii=False))
print(f"wrote {META/'prompts.json'}")

bare = [i for i in range(256)
        if FULL[str(i)].strip().lower() == CLASSES[i].strip().lower()]
print("bare-name prompts:", bare)

In [ ]:
import shutil, json
shutil.move("/workspace/uec256_prompts.json", str(META/"uec256_prompts.json"))

FULL = json.loads((META/"uec256_prompts.json").read_text())
assert len(FULL) == 256
assert all(str(i) in FULL for i in range(256))

out = {str(i): {"name": CLASSES[i], "prompt": FULL[str(i)]} for i in range(256)}
(META/"prompts.json").write_text(json.dumps(out, indent=2, ensure_ascii=False))
print(f"wrote {META/'prompts.json'}  ({len(out)} entries)")

bare = [i for i in range(256)
        if FULL[str(i)].strip().lower() == CLASSES[i].strip().lower()]
print("bare-name prompts:", bare)

for i in (0, 34, 53, 138, 139, 255):
    print(f"{i:>3} {CLASSES[i]:<26} {FULL[str(i)][:70]}")

In [ ]:
BUDGET = 7680          # 30 per class average, ~35% of the 22,041 real training instances

cm_val = np.load(META/"A_align_val_confusion_pooled.npy")
row_err = cm_val.sum(axis=1) - np.diag(cm_val)

# arm B: uniform
alloc_B = np.full(N_CLASSES, BUDGET // N_CLASSES, dtype=int)
for i in range(BUDGET - int(alloc_B.sum())):
    alloc_B[i] += 1

# arm C: proportional to pooled validation confusion mass
w = row_err / row_err.sum()
alloc_C = np.floor(w * BUDGET).astype(int)
for idx in np.argsort(-(w*BUDGET - alloc_C))[:BUDGET - int(alloc_C.sum())]:
    alloc_C[idx] += 1

assert alloc_B.sum() == alloc_C.sum() == BUDGET
print(f"budget {BUDGET}   per class uniform {alloc_B[0]}")
print(f"corr(B,C) {np.corrcoef(alloc_B, alloc_C)[0,1]:.3f}   "
      f"(B is constant, so this is undefined/0 by construction)")
print(f"C: min {alloc_C.min()}  max {alloc_C.max()}  "
      f"zero-allocation classes {(alloc_C==0).sum()}")

print(f"\n{'class':<34} {'real':>5} {'err':>5} {'B':>5} {'C':>5}")
for i in np.argsort(-alloc_C)[:15]:
    print(f"{CLASSES[i][:34]:<34} {train_counts[i]:>5} {row_err[i]:>5} "
          f"{alloc_B[i]:>5} {alloc_C[i]:>5}")
print("...")
for i in np.argsort(alloc_C)[:8]:
    print(f"{CLASSES[i][:34]:<34} {train_counts[i]:>5} {row_err[i]:>5} "
          f"{alloc_B[i]:>5} {alloc_C[i]:>5}")

In [ ]:
BUDGET = 5376          # 21 per class uniform; 24% of 22,041 real, matching Food101-LT's 25%

In [ ]:
pool_need = {i: int(max(alloc_B[i], alloc_C[i])) for i in range(N_CLASSES)}
(META/"pool_alloc.json").write_text(json.dumps(pool_need, indent=2))
print(f"pool {sum(pool_need.values())}   budget per arm {BUDGET}   "
      f"overhead {100*sum(pool_need.values())/BUDGET - 100:.0f}%")

In [ ]:
%%writefile /workspace/gen_synth_uec.py
# coding: utf-8
"""Generate the shared synthetic pool for UEC-256 arms B, C and D."""
import json
from pathlib import Path
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

META  = Path("/workspace/uec256_meta")
POOL  = Path("/workspace/uec256_synth"); POOL.mkdir(exist_ok=True)
CACHE = "/workspace/hf_cache"
STEPS, GUIDANCE, BATCH, SEED = 50, 2.0, 8, 42

need    = {int(k): v for k, v in json.loads((META/"pool_alloc.json").read_text()).items()}
prompts = json.loads((META/"prompts.json").read_text())

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16,
    safety_checker=None, cache_dir=CACHE).to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

g = torch.Generator("cuda")
total = sum(need.values()); done_all = 0

for ci in sorted(need):
    n = need[ci]
    d = POOL/f"{ci:03d}"; d.mkdir(exist_ok=True)
    have = len(list(d.glob("*.jpg")))
    if have >= n:
        done_all += n
        print(f"[{ci:>3}/255] {prompts[str(ci)]['name'][:28]:<28} {have} already", flush=True)
        continue

    prompt = f"A photo of {prompts[str(ci)]['prompt']}."
    made = have
    while made < n:
        k = min(BATCH, n - made)
        g.manual_seed(SEED * 1000000 + ci * 1000 + made)
        imgs = pipe([prompt]*k, num_inference_steps=STEPS,
                    guidance_scale=GUIDANCE, generator=g).images
        for im in imgs:
            im.resize((512, 512)).save(d/f"{made:05d}.jpg", quality=95)
            made += 1
    done_all += made
    print(f"[{ci:>3}/255] {prompts[str(ci)]['name'][:28]:<28} {made}   "
          f"({100*done_all/total:.1f}% of pool)", flush=True)

print("pool complete")

In [ ]:
cd /workspace && nohup python gen_synth_uec.py > gen_uec.log 2>&1 &
tail -f gen_uec.log

In [ ]:
!cd /workspace && nohup python gen_synth_uec.py > gen_uec.log 2>&1 &

In [ ]:
have = {ci: len(list((POOL/f"{ci:03d}").glob("*.jpg"))) for ci in range(N_CLASSES)}
need = {int(k): v for k, v in json.loads((META/"pool_alloc.json").read_text()).items()}
short = {ci: (have[ci], need[ci]) for ci in range(N_CLASSES) if have[ci] < need[ci]}
print(f"pool {sum(have.values())} / {sum(need.values())}   short in {len(short)} classes")

import matplotlib.pyplot as plt
for ci in (1, 133, 39, 66, 75, 76):
    fs = sorted((POOL/f"{ci:03d}").glob("*.jpg"))[:4]
    fig, ax = plt.subplots(1, 4, figsize=(12,3))
    for a, f in zip(np.atleast_1d(ax), fs):
        a.imshow(Image.open(f)); a.axis("off")
    fig.suptitle(f"{ci}: {CLASSES[ci]}", fontsize=10); plt.show()

In [ ]:
@torch.no_grad()
def embed(paths_or_inst, is_real, model, bs=64):
    """Return L2-normalised features [N, D]."""
    feats = []
    for i in range(0, len(paths_or_inst), bs):
        batch = paths_or_inst[i:i+bs]
        ims = []
        for item in batch:
            if is_real:
                f, box, _, _ = item
                im = Image.open(f).convert("RGB").crop(box)
            else:
                im = Image.open(item).convert("RGB")
            ims.append(eval_tf(im))
        x = torch.stack(ims).to(DEVICE)
        with torch.amp.autocast("cuda"):
            z = model(x)
        feats.append(F.normalize(z.float(), dim=1).cpu())
    return torch.cat(feats)

# feature extractor: the arm-A backbone with the classifier head removed
enc = build_swin(N_CLASSES, "align")
enc.load_state_dict(torch.load(ARMS/"A_align_s42.pth", map_location="cpu"))
enc.head = nn.Identity()
enc = enc.to(DEVICE).eval()

scores = {}
for ci in range(N_CLASSES):
    real = [x for x in TR_INST if x[2] == ci]
    synth = sorted((POOL/f"{ci:03d}").glob("*.jpg"))
    zr = embed(real, True, enc)
    centroid = F.normalize(zr.mean(0, keepdim=True), dim=1)
    zs = embed(synth, False, enc)
    sim = (zs @ centroid.T).squeeze(1)
    scores[ci] = {"files": [str(p) for p in synth],
                  "sim": sim.tolist(),
                  "mean": float(sim.mean()),
                  "real_spread": float((zr @ centroid.T).mean())}
    if ci % 25 == 0:
        print(f"{ci:>3} {CLASSES[ci][:26]:<26} mean {sim.mean():.3f}  "
              f"real {scores[ci]['real_spread']:.3f}", flush=True)

json.dump(scores, open(META/"fidelity.json", "w"))
print("\nwrote fidelity.json")

m = np.array([scores[ci]["mean"] for ci in range(N_CLASSES)])
order = np.argsort(m)
print("\nlowest fidelity:")
for i in order[:12]: print(f"  {CLASSES[i][:32]:<32} {m[i]:.3f}")
print("highest fidelity:")
for i in order[-8:]: print(f"  {CLASSES[i][:32]:<32} {m[i]:.3f}")

In [ ]:
m  = np.array([scores[ci]["mean"] for ci in range(N_CLASSES)])
rs = np.array([scores[ci]["real_spread"] for ci in range(N_CLASSES)])
ratio = m / rs

print(f"raw   min {m.min():.3f}  median {np.median(m):.3f}  max {m.max():.3f}")
print(f"ratio min {ratio.min():.3f}  median {np.median(ratio):.3f}  max {ratio.max():.3f}")
print(f"classes with ratio < 0.2: {(ratio<0.2).sum()}")
print(f"spearman(fidelity, val errors): "
      f"{__import__('scipy.stats', fromlist=['s']).spearmanr(m, row_err).correlation:.3f}")

In [ ]:
from scipy.stats import spearmanr

m  = np.array([scores[ci]["mean"] for ci in range(N_CLASSES)])
rs = np.array([scores[ci]["real_spread"] for ci in range(N_CLASSES)])
ratio = np.clip(m / rs, 0, None)          # negative fidelity -> zero weight

w = ratio / ratio.sum()
alloc_D = np.floor(w * BUDGET).astype(int)
for idx in np.argsort(-(w*BUDGET - alloc_D))[:BUDGET - int(alloc_D.sum())]:
    alloc_D[idx] += 1

assert alloc_D.sum() == BUDGET
print(f"D: min {alloc_D.min()}  max {alloc_D.max()}  zeros {(alloc_D==0).sum()}")
print(f"corr(C,D) {np.corrcoef(alloc_C, alloc_D)[0,1]:.3f}   "
      f"spearman {spearmanr(alloc_C, alloc_D).correlation:.3f}")

print(f"\n{'class':<32} {'err':>4} {'fid':>6} {'B':>4} {'C':>4} {'D':>4}")
for i in np.argsort(-alloc_D)[:10]:
    print(f"{CLASSES[i][:32]:<32} {row_err[i]:>4} {ratio[i]:>6.3f} "
          f"{alloc_B[i]:>4} {alloc_C[i]:>4} {alloc_D[i]:>4}")
print("...")
for i in np.argsort(alloc_D)[:10]:
    print(f"{CLASSES[i][:32]:<32} {row_err[i]:>4} {ratio[i]:>6.3f} "
          f"{alloc_B[i]:>4} {alloc_C[i]:>4} {alloc_D[i]:>4}")

In [ ]:
def write_list(alloc, name):
    lines = []
    for ci in range(N_CLASSES):
        if alloc[ci] == 0: continue
        s = scores[ci]
        idx = np.argsort(-np.array(s["sim"]))[:alloc[ci]]   # best-scoring first
        assert len(idx) == alloc[ci], (ci, len(s["files"]), alloc[ci])
        lines += [f"{s['files'][j]}\t{ci}" for j in idx]
    p = META/f"synth_{name}.txt"
    p.write_text("\n".join(lines) + "\n")
    print(f"{name}: {len(lines)} images -> {p}")
    return p

SYNTH_B = write_list(alloc_B, "B")
SYNTH_C = write_list(alloc_C, "C")
SYNTH_D = write_list(alloc_D, "D")

In [ ]:
pool_have = np.array([len(list((POOL/f"{ci:03d}").glob("*.jpg")))
                      for ci in range(N_CLASSES)])

def cap_to_pool(alloc, have):
    a = np.minimum(alloc, have).astype(int)
    deficit = int(alloc.sum() - a.sum())
    if deficit == 0:
        return a
    # give the shortfall to classes with spare capacity, in fidelity order
    spare = have - a
    for ci in np.argsort(-ratio):
        if deficit == 0: break
        take = min(spare[ci], deficit)
        a[ci] += take; deficit -= take
    assert deficit == 0, f"pool too small by {deficit}"
    return a

alloc_D = cap_to_pool(alloc_D, pool_have)
alloc_C = cap_to_pool(alloc_C, pool_have)
alloc_B = cap_to_pool(alloc_B, pool_have)
assert alloc_B.sum() == alloc_C.sum() == alloc_D.sum() == BUDGET
print("capped. D:", alloc_D.min(), alloc_D.max(), "zeros", (alloc_D==0).sum())
print(f"corr(C,D) {np.corrcoef(alloc_C, alloc_D)[0,1]:.3f}")

In [ ]:
print("pool total:", pool_have.sum(), " BUDGET:", BUDGET)
for nm, a in (("B", alloc_B), ("C", alloc_C), ("D", alloc_D)):
    print(f"{nm}: sum {a.sum():>6}  (target {BUDGET}, diff {a.sum()-BUDGET:+})")

In [ ]:
pool_have = np.array([len(list((POOL/f"{ci:03d}").glob("*.jpg")))
                      for ci in range(N_CLASSES)])
print("pool total:", pool_have.sum(), " BUDGET:", BUDGET)
assert pool_have.sum() >= BUDGET, "pool smaller than budget"

def cap_to_pool(alloc, have, priority):
    target = int(alloc.sum())
    a = np.minimum(alloc, have).astype(int)
    deficit = target - int(a.sum())
    for ci in priority:
        if deficit <= 0: break
        take = int(min(have[ci] - a[ci], deficit))
        a[ci] += take; deficit -= take
    assert deficit == 0, f"cannot place {deficit} images"
    assert (a <= have).all()
    return a

by_fidelity = np.argsort(-ratio)          # spare capacity goes to best classes first
alloc_B = cap_to_pool(alloc_B, pool_have, by_fidelity)
alloc_C = cap_to_pool(alloc_C, pool_have, by_fidelity)
alloc_D = cap_to_pool(alloc_D, pool_have, by_fidelity)

for nm, a in (("B", alloc_B), ("C", alloc_C), ("D", alloc_D)):
    print(f"{nm}: sum {a.sum()}  min {a.min()}  max {a.max()}  zeros {(a==0).sum()}")
assert alloc_B.sum() == alloc_C.sum() == alloc_D.sum() == BUDGET

print(f"\ncorr(B,C) {np.corrcoef(alloc_B, alloc_C)[0,1]:.3f}")
print(f"corr(B,D) {np.corrcoef(alloc_B, alloc_D)[0,1]:.3f}")
print(f"corr(C,D) {np.corrcoef(alloc_C, alloc_D)[0,1]:.3f}")

In [ ]:
BUDGET = 5376

pool_have = np.array([len(list((POOL/f"{ci:03d}").glob("*.jpg")))
                      for ci in range(N_CLASSES)])
cm_val  = np.load(META/"A_align_val_confusion_pooled.npy")
row_err = cm_val.sum(axis=1) - np.diag(cm_val)

m     = np.array([scores[ci]["mean"] for ci in range(N_CLASSES)])
rs    = np.array([scores[ci]["real_spread"] for ci in range(N_CLASSES)])
ratio = np.clip(m / rs, 0, None)

def largest_remainder(weights, budget):
    w = weights / weights.sum()
    a = np.floor(w * budget).astype(int)
    for idx in np.argsort(-(w*budget - a))[:budget - int(a.sum())]:
        a[idx] += 1
    return a

def cap_to_pool(alloc, have, priority):
    target = int(alloc.sum())
    a = np.minimum(alloc, have).astype(int)
    deficit = target - int(a.sum())
    for ci in priority:
        if deficit <= 0: break
        take = int(min(have[ci] - a[ci], deficit))
        a[ci] += take; deficit -= take
    assert deficit == 0, f"cannot place {deficit}"
    return a

by_fidelity = np.argsort(-ratio)

alloc_B = largest_remainder(np.ones(N_CLASSES), BUDGET)   # uniform
alloc_C = largest_remainder(row_err.astype(float), BUDGET) # confusion
alloc_D = largest_remainder(ratio, BUDGET)                 # fidelity

alloc_B = cap_to_pool(alloc_B, pool_have, by_fidelity)
alloc_C = cap_to_pool(alloc_C, pool_have, by_fidelity)
alloc_D = cap_to_pool(alloc_D, pool_have, by_fidelity)

for nm, a in (("B", alloc_B), ("C", alloc_C), ("D", alloc_D)):
    print(f"{nm}: sum {a.sum()}  min {a.min()}  max {a.max()}  zeros {(a==0).sum()}")
assert alloc_B.sum() == alloc_C.sum() == alloc_D.sum() == BUDGET

print(f"\ncorr(B,C) {np.corrcoef(alloc_B, alloc_C)[0,1]:.3f}")
print(f"corr(B,D) {np.corrcoef(alloc_B, alloc_D)[0,1]:.3f}")
print(f"corr(C,D) {np.corrcoef(alloc_C, alloc_D)[0,1]:.3f}")

In [ ]:
SYNTH_B = write_list(alloc_B, "B")
SYNTH_C = write_list(alloc_C, "C")
SYNTH_D = write_list(alloc_D, "D")

In [ ]:
for s in (42, 1, 2):
    run_arm(f"B_align_s{s}", init="align", synth=SYNTH_B, alloc="uniform",
            expect_synth=BUDGET, seed=s)
    run_arm(f"C_align_s{s}", init="align", synth=SYNTH_C, alloc="targeted",
            expect_synth=BUDGET, seed=s)
    run_arm(f"D_align_s{s}", init="align", synth=SYNTH_D, alloc="fidelity",
            expect_synth=BUDGET, seed=s)